# Mini-project 2: Solar micro-grid dispatch planner
In this notebook, we model and solve daily energy dispatch from solar panels and batteries for a rural health centre in Kasese.

In [1]:
import numpy as np
import scipy.linalg
import scipy.optimize
import statistics as st
import matplotlib.pyplot as plt
import timeit
import os

# Import our domain logic classes
from src.solar_class import MicroGrid, HybridMicroGrid

# Set seed for reproducibility
rng = np.random.default_rng(42)

## Well-posedness
We check if the coefficient matrix is well-posed by calculating its determinant and condition number.
Constraints:
- 3x + 2y = D1
- 4x + 1y = D2

Explanation:
- Determinant: A non-zero determinant means the coefficient matrix is invertible. This guarantees that for valid daytime and critical loads, there is exactly one unique mathematical solution for x and y.
- Condition Number: Measures how sensitive the solution is to small changes in demand. A low condition number close to 1 means the system is stable and well-conditioned; small errors in demand estimates will not cause chaotic swings in the calculated dispatch.

In [2]:
grid = MicroGrid()
det, cond = grid.check_well_posed()

print(f"Determinant: {det:.2f}")
print(f"Condition Number: {cond:.2f}")



## Input modes
We support interactive input (commented out for automated execution) and CSV loading.

In [3]:
# Interactive Mode Example (uncomment to run manually)
# d1_interactive, d2_interactive = MicroGrid.get_interactive_demand()
# sol = grid.solve_day(d1_interactive, d2_interactive)
# print(f"Interactive Solution - Solar: {sol[0]:.2f} kWh, Battery: {sol[1]:.2f} kWh")

# Generate and load CSV data
# We intentionally inject edge-case anomalies to trigger infeasible dispatches later on.
csv_file = MicroGrid.generate_csv_data(days=30, seed=42)
D_matrix = MicroGrid.load_csv_data(csv_file)
print(f"Loaded {D_matrix.shape[1]} days of demand data from {csv_file}.")

## Vectorized vs loop solving
We solve the 30 days of demand first in a Python loop, and then using a single vectorized call.

Difference:
The vectorized approach is faster. Passing a 2x30 right-hand side directly to scipy.linalg.solve pushes execution down into compiled, optimized BLAS and LAPACK routines that solve the batch simultaneously. The loop approach incurs Python interpreter overhead on each day.

In [4]:
def solve_loop(D_matrix):
    solutions = []
    for i in range(D_matrix.shape[1]):
        sol = grid.solve_day(D_matrix[0, i], D_matrix[1, i])
        solutions.append(sol)
    return np.array(solutions).T

def solve_vectorized(D_matrix):
    return grid.solve_vectorized(D_matrix)

# Correctness check
sol_loop = solve_loop(D_matrix)
sol_vec = solve_vectorized(D_matrix)
np.testing.assert_allclose(sol_loop, sol_vec)

# Timing
loop_time = timeit.timeit(lambda: solve_loop(D_matrix), number=1000)
vec_time = timeit.timeit(lambda: solve_vectorized(D_matrix), number=1000)

print(f"Loop Time (1000 iterations): {loop_time:.4f} seconds")
print(f"Vectorized Time (1000 iterations): {vec_time:.4f} seconds")



## Infeasible days
Physically infeasible days occur if the mathematical solution requires negative solar or battery usage.
We implement a Non-Negative Least Squares (NNLS) fallback strategy.

Handling strategy:
We switch to a Non-Negative Least Squares solver (scipy.optimize.nnls). This forces x >= 0 and y >= 0, minimizing squared constraint violations when demand cannot be met exactly.

In [5]:
# Extract initial mathematical solutions
solutions = solve_vectorized(D_matrix)
solar_usage = solutions[0, :].copy()
battery_usage = solutions[1, :].copy()

infeasible_days = np.where((solar_usage < 0) | (battery_usage < 0))[0]

print(f"Detected {len(infeasible_days)} physically infeasible days.")

# Fallback: recalculate using NNLS
for day_idx in infeasible_days:
    d1 = D_matrix[0, day_idx]
    d2 = D_matrix[1, day_idx]
    
    # Standard solution (mathematical)
    orig_sol = grid.solve_day(d1, d2)
    
    # NNLS solution (physical constraints enforced)
    nnls_sol = grid.solve_day_nnls(d1, d2)
    
    solar_usage[day_idx] = nnls_sol[0]
    battery_usage[day_idx] = nnls_sol[1]
    
    print(f"Day {day_idx+1}: Demands ({d1:.1f}, {d2:.1f}) -> Original: ({orig_sol[0]:.1f}, {orig_sol[1]:.1f}) | NNLS Fixed: ({nnls_sol[0]:.1f}, {nnls_sol[1]:.1f})")



## Statistics
We compute descriptive statistics for usage arrays to evaluate volatility.

Volatility evaluation:
Battery usage has a substantially higher variance than solar usage, making it the more volatile power source in this micro-grid setup.

In [6]:
solar_list = solar_usage.tolist()
battery_list = battery_usage.tolist()

stats = {
    'Solar': {
        'Mean': st.mean(solar_list),
        'Variance': st.variance(solar_list),
        'StdDev': st.stdev(solar_list)
    },
    'Battery': {
        'Mean': st.mean(battery_list),
        'Variance': st.variance(battery_list),
        'StdDev': st.stdev(battery_list)
    }
}

for src, metrics in stats.items():
    print(f"{src} Usage -> Mean: {metrics['Mean']:.2f}, Variance: {metrics['Variance']:.2f}, StdDev: {metrics['StdDev']:.2f}")



## Cost model
Daily and monthly cost estimation based on standard unit rates (UGX 150/kWh for solar and UGX 450/kWh for battery).

In [7]:
SOLAR_RATE = 150
BATTERY_RATE = 450

daily_cost = (solar_usage * SOLAR_RATE) + (battery_usage * BATTERY_RATE)
monthly_cost = np.sum(daily_cost)

print(f"Total Monthly Cost: UGX {monthly_cost:,.2f}")

## Visualization
Plotting daily energy usage and cost.

In [8]:
days = np.arange(1, 31)

fig, ax1 = plt.subplots(figsize=(12, 6))

ax1.bar(days, solar_usage, label='Solar (x)', color='gold')
ax1.bar(days, battery_usage, bottom=solar_usage, label='Battery (y)', color='darkorange')
ax1.set_xlabel('Day')
ax1.set_ylabel('Energy Usage (kWh)')
ax1.set_title('Daily Energy Dispatch and Cost')
ax1.legend(loc='upper left')

# Secondary axis
ax2 = ax1.twinx()
ax2.plot(days, daily_cost, color='green', marker='o', linestyle='-', linewidth=2, label='Daily Cost (UGX)')
ax2.set_ylabel('Cost (UGX)')
ax2.legend(loc='upper right')

plt.tight_layout()
plt.savefig('microgrid_dispatch.png')
plt.show()

## Extension: Hybrid micro-grid and linear dependence
Adding a diesel generator (z) and testing linear dependence of a third constraint.

Discussion:
When the new equation is linearly dependent on existing ones, the determinant falls to 0 (system becomes singular). The constraints do not provide enough unique information to solve for all 3 variables (x, y, z), leading to infinite solutions or no solution depending on the right-hand side.

In [9]:
print("--- Extension: Hybrid Micro-Grid ---")
# Suppose our third constraint is: x + y + z = D3
hybrid_grid = HybridMicroGrid()
det_hybrid, cond_hybrid = hybrid_grid.check_well_posed_hybrid()
print(f"Independent Hybrid System - Determinant: {det_hybrid:.2f}")

# What if the new equation is linearly dependent?
# e.g., equation 3 is simply equation 1 + equation 2
# 3x + 2y + 0z = D1
# 4x + 1y + 0z = D2
# 7x + 3y + 0z = D3
dependent_hybrid = HybridMicroGrid(new_equation=[7.0, 3.0, 0.0])
det_dep, _ = dependent_hybrid.check_well_posed_hybrid()
print(f"Dependent Hybrid System - Determinant: {det_dep:.2f}")



## Extension: Sensitivity analysis
Perturbing D1 and D2 by +/-5% to see the impact on x and y, and relating it back to the condition number.

Relation to condition number:
Because the condition number is low and close to 1, the relative error in the solution is strictly bounded by a small multiple of the relative error in the input. A 5% perturbation in demand led to deviations strictly within ~5%, illustrating the stability of a well-conditioned system.

In [10]:
print("\n--- Extension: Sensitivity Analysis ---")

base_d1 = 120.0
base_d2 = 90.0
base_sol = grid.solve_day(base_d1, base_d2)
base_x, base_y = base_sol

# Monte Carlo: 1000 draws perturbing D1 and D2 by uniform ±5%
n_draws = 1000
d1_samples = base_d1 * rng.uniform(0.95, 1.05, n_draws)
d2_samples = base_d2 * rng.uniform(0.95, 1.05, n_draws)

# Vectorized solving for all samples
D_samples = np.vstack([d1_samples, d2_samples])
sols = grid.solve_vectorized(D_samples)

x_samples = sols[0, :]
y_samples = sols[1, :]

# Calculate max percentage deviation
max_x_dev = np.max(np.abs(x_samples - base_x)) / base_x * 100
max_y_dev = np.max(np.abs(y_samples - base_y)) / base_y * 100

print(f"Base Demands (D1=120, D2=90) -> Solar(x)={base_x:.2f}, Battery(y)={base_y:.2f}")
print(f"After ±5% perturbation over {n_draws} samples:")
print(f"Max deviation in Solar (x): {max_x_dev:.2f}%")
print(f"Max deviation in Battery (y): {max_y_dev:.2f}%")



## Conclusions

The dispatch of a solar micro-grid can be modeled and solved as a system of linear equations. Monitoring the condition number confirms the system is well-posed and stable against fluctuations in demand. Detecting physical infeasibilities (like negative power draw) is required for real-world operation, allowing for adjustments like fallback non-negative least-squares optimization.

## References

- System of linear equations: https://en.wikipedia.org/wiki/System_of_linear_equations
- Determinant: https://en.wikipedia.org/wiki/Determinant
- Condition number: https://en.wikipedia.org/wiki/Condition_number
- Monte Carlo method (for sensitivity analysis): https://en.wikipedia.org/wiki/Monte_Carlo_method